In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('cifar-100-image-classification-challenge')

print("Path to competition files:", path)

In [ ]:
import os
import pickle
from torchvision import datasets, transforms
from torch.utils.data import DataLoader 
from PIL import Image


with open(os.path.join(path, "train"), "rb") as f: train_data = pickle.load(f, encoding="bytes")
with open(os.path.join(path, "test"), "rb") as f: test_data = pickle.load(f, encoding="bytes")







In [ ]:


#Make Image and Label variables for all images and labels of datset with correct shape and type
images = train_data[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # Convert to NCH
labels = train_data[b"fine_labels"]
print("Images shape:", images.shape)
print("Labels shape:", len(labels))

#Since there are no test labels, we will just create a dummy array of zeros for the test labels
test_images = test_data[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # Convert to NCH
test_labels = [0] * len(test_images)  # Create a dummy list of zeros

print("Test Images shape:", test_images.shape)



In [ ]:
# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [ ]:
from torch.utils.data import Dataset
class CIFAR100Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
train_dataset = CIFAR100Dataset(images, labels, transform=transform)
#test_dataset = CIFAR100Dataset(test_images, test_labels, transform=transform)
#train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
#test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Since the test lables are not provided, we will not be able to evaluate the model on the test set. We will just train the model on the training set and then make predictions on the test set without evaluating them.
from torch.utils.data import random_split
train_size = int(0.7 * len(train_dataset))
test_size = len(train_dataset) - train_size

train_dataset, test_dataset = random_split(train_dataset, [train_size, test_size])
print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
#Model CNN Design
import torch
import torch.nn as nn
import torch.nn.functional as F
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 100)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    




In [ ]:
# Try a prebuirlt model like ResNet18
from torchvision import models
resnet50 = models.resnet50(pretrained=True)
# Modify the final layer to match the number of classes in CIFAR-100
resnet50.fc = nn.Linear(resnet50.fc.in_features, 100)

In [ ]:
# Training code
import torch.optim as optim
#model = SimpleCNN()
# Try using ResNet18 instead of SimpleCNN
model = resnet50

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(10):  # Just one epoch for demonstration
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")   
    
print("Training complete for 10 epochs.")

#Evaluate on test set carved from training set since test labels are not provided

#Accuracy of Model on test set carved from training set
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Accuracy on test set: {100 * correct / total:.2f}%")

        

        
        # Write predictions to a CSV file
        #with open("predictions.csv", "a") as f:
        #    for pred in predicted:
        #        f.write(f"{pred.item()}\n")
#print("Predictions saved to predictions.csv")








In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset  # Fixed imports
from torchvision import models, transforms

# 1. DATA LOADING HELPER
def unpickle(file):
    with open(file, 'rb') as fo:
        dict_data = pickle.load(fo, encoding='bytes')
    return dict_data

# 2. DATASET CLASS
class CIFAR100Dataset(Dataset):
    def __init__(self, data_dict, transform=None, is_test=False):
        super().__init__()
        # Reshape flattened data to (N, H, W, C)
        self.images = data_dict[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.transform = transform
        self.is_test = is_test
        if not is_test:
            self.labels = data_dict[b'fine_labels']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        
        if self.is_test:
            # Kaggle usually expects a specific ID. 
            # If the test set has b'ids' or b'filenames', use those, 
            # otherwise use the index.
            return img, idx 
        return img, self.labels[idx]

# 3. CONFIGURATION
# 'path' should be the variable from your kagglehub download
DATA_PATH = path 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128 # Increased for better stability
num_epochs = 20  

# 4. TRANSFORMS
stats = ((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
train_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

valid_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

# 5. PREPARE DATA
train_raw = unpickle(os.path.join(DATA_PATH, 'train'))
test_raw = unpickle(os.path.join(DATA_PATH, 'test'))

full_train_ds = CIFAR100Dataset(train_raw, transform=train_tfms)
full_valid_ds = CIFAR100Dataset(train_raw, transform=valid_tfms)

# Manual Shuffle and Split
indices = list(range(len(full_train_ds)))
np.random.seed(42) # For reproducibility
np.random.shuffle(indices)
split = int(0.1 * len(full_train_ds)) # 10% for validation is enough for 50k images
train_idx, valid_idx = indices[split:], indices[:split]

train_loader = DataLoader(Subset(full_train_ds, train_idx), batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(Subset(full_valid_ds, valid_idx), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(CIFAR100Dataset(test_raw, transform=valid_tfms, is_test=True), batch_size=batch_size, shuffle=False)

# 6. MODEL: ResNet-50 Optimized for 32x32
model = models.resnet50(weights=None)
# Modifying the first conv layer because the images are small (32x32)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity() # Removing maxpool preserves spatial resolution for small images
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# 7. TRAINING LOOP
print(f"Training on {device}...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in valid_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            _, pred = torch.max(out, 1)
            correct += (pred == labels).sum().item()
    
    acc = 100 * correct / len(valid_idx)
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}%")

# 8. FINAL PREDICTIONS
print("Saving predictions.csv...")
model.eval()
results = []
with torch.no_grad():
    for imgs, idxs in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        _, pred = torch.max(out, 1)
        # Convert to CPU and append
        for i, p in zip(idxs.numpy(), pred.cpu().numpy()):
            results.append({"Id": i, "label": p})

pd.DataFrame(results).to_csv('predictions.csv', index=False)
print("Done!")

Training on cpu...
Epoch 1/20 | Loss: 4.0492 | Val Acc: 14.98%
